In [ ]:
import pymupdf
import os
import sys
from cloudinary import uploader, config
import json
from typing import Dict
import re

# make sure the project root is on the import path
sys.path.append('/home/inacio/script-clima-amazonia')

from helpers.texts.dictClasses import dict_classes
from helpers.texts.slugify import slugify
from helpers.images.analysis import img_analysis
from helpers.images.current_conditions import img_current_conditions
from helpers.images.multimodel import img_multimodel
from helpers.images.reference import img_reference
from helpers.images.categorization import img_categorization
from helpers.images.anomaly import img_anomaly

In [9]:
number = "20260304"
pathPT = f"/home/inacio/script-clima-amazonia/pdfs/BHA_PT_{number}.pdf"
pathEN = f"/home/inacio/script-clima-amazonia/pdfs/BHA_EN_{number}.pdf"
pathES = f"/home/inacio/script-clima-amazonia/pdfs/BHA_ES_{number}.pdf"

# Text

In [18]:
def date_to_en(date):
    meses = {
        "janeiro": "January",
        "fevereiro": "February",
        "março": "March",
        "marco": "March",
        "abril": "April",
        "maio": "May",
        "junho": "June",
        "julho": "July",
        "agosto": "August",
        "setembro": "September",
        "outubro": "October",
        "novembro": "November",
        "dezembro": "December"
    }
    mes = date.split()[2]
    dia = date.split()[0]
    ano = date.split()[-1]
    mes_en = meses[mes]
    date_en = f'{mes_en} {dia}, {ano}'
    return date_en

def date_to_es(date):
    meses = {
        "janeiro": "enero",
        "fevereiro": "febrero",
        "março": "marzo",
        "marco": "marzo",
        "abril": "abril",
        "maio": "mayo",
        "junho": "junio",
        "julho": "julio",
        "agosto": "agosto",
        "setembro": "septiembre",
        "outubro": "octubre",
        "novembro": "noviembre",
        "dezembro": "diciembre"
    }
    mes = date.split()[2]
    dia = date.split()[0]
    ano = date.split()[-1]
    mes_es = meses[mes]
    date_es = f'{dia} de {mes_es} de {ano}'
    return date_es

def get_meta(doc, bulletin_dict):
    # Page 1
    page = doc.load_page(0)
    text = page.get_text()
    text_lines = text.splitlines()
    number = text_lines[-1].split()[3]
    date = " ".join(text_lines[-1].split()[5:])

    meta = {
        "meta": {
        "pt": {
            "doi": f'10.61818/029106{number}',
            "issn": "2965-0291",
            "volume": "06",
            "date": date
        },
        "en": {
            "doi": f'10.61818/785704{number}',
            "issn": "2965-7857",
            "volume": "04",
            "date": date_to_en(date)
        },
        "es": {
            "doi": f'10.61818/770904{number}',
            "issn": "2965-7709",
            "volume": "04",
            "date": date_to_es(date)
        },
         "number": number
    }
        }
    bulletin_dict.update(meta)
    return bulletin_dict

def get_current_conditions(pathPT, pathEN, pathES, bulletin_dict):
    doc = pymupdf.open(pathPT)     
    page = doc.load_page(3)
    textPT = page.get_text()
    textPT = re.sub(r'\n+', ' ', textPT)
    textPT = re.sub(r'\s+', ' ', textPT)
    textPT = textPT.split("Condições atuais ")[1]
    textPT = textPT.split("1 Abacaxis ")[0].rstrip()
    # EN
    doc = pymupdf.open(pathEN)     
    page = doc.load_page(3)
    textEN = page.get_text()
    textEN = re.sub(r'\n+', ' ', textEN)
    textEN = re.sub(r'\s+', ' ', textEN)
    textEN = textEN.split("Current conditions ")[1]
    textEN = textEN.split("1 Abacaxis ")[0].rstrip()
    # ES
    doc = pymupdf.open(pathES)     
    page = doc.load_page(3)
    textES = page.get_text()
    textES = re.sub(r'\n+', ' ', textES)
    textES = re.sub(r'\s+', ' ', textES)
    textES = textES.split("Condiciones actuales ")[1]
    textES = textES.split("1 Abacaxis ")[0].rstrip()
    
    current_conditions = {
        "pt": textPT,
        "en": textEN,
        "es": textES
    }
    
    bulletin_dict['current_conditions'] = current_conditions
    return bulletin_dict 


def extract_fields(text: str) -> Dict[str, str]:

    climatologia = re.search(r"entre\s+(\d+\s+e\s+\d+\s+mm)", text).group(1)
    min = climatologia.split()[0]
    max = climatologia.split()[2]
    observados = re.search(r"foram\s+observados\s+(\d+\s+mm)", text)
    anomalia = re.search(r"valor\s+de\s+([-\d\.]+)", text)
    classification = re.search(r"classifica\s+a\s+bacia\s+em\s+condição\s+de\s+([^\.]+)", text)
    prognostico = re.search(r"sugere\s+um\s+comportamento\s+([^\.]+)", text)
    trend = text.split("comportamento climático indica ")
    trend = trend[1].split(" ",1)[0]


    return {
        "min": min,
        "max": max,
        "observados": observados.group(1) if observados else None,
        "anomalia": anomalia.group(1) if anomalia else None,
        "classification": classification.group(1).strip() if classification else None,
        "prognostico": prognostico.group(1).strip() if prognostico else None,
        "trend": trend
    }


def dict_trend(trend, lang):
    d = {
        "elevacao": {
            "en": "increase",
            "es": "elevación"
        },
        "manutencao": {
            "en": "increase",
            "es": "maintenance"
        },
        "reducao": {
            "en": "decrease",
            "es": "reducción"
        }
    }
    return d[trend][lang]

def dict_prognostico(prognostico, lang):
    d = {
        "muito-seco-ou-tendencia-a-extremamente-seco": {
            "en": "very dry behavior or a tending to be extremely dry",
            "es": "muy seco o con tendencia a ser extremadamente seco"
        },
        "muito-seco-ou-tendencia-a-muito-seco": {
            "en": "very dry behavior or a tending to be very dry",
            "es": "muy seco o con tendencia a ser muy seco"
        },
        "seco-ou-tendencia-a-muito-seco": {
            "en": "dry behavior or a tending to be very dry",
            "es": "seco o con tendencia a ser muy seco"
        },
        "chuvoso-ou-tendencia-a-chuvoso": {
            "en": "rainy behavior or a tending to be rainy",
            "es": "lluvioso o propenso a lluvioso."
        },
        "proximo-da-normalidade-ou-tendencia-a-chuvoso": {
            "en": "behavior close to normality or a tending to be rainy",
            "es": "cerca de la normalidad o con tendencia a lluvioso"
        },
        "proximo-da-normalidade-ou-tendencia-a-seco": {
            "en": "behavior close to normality or a tending to be dry",
            "es": "cerca de la normalidad o con tendencia a ser seco"
        },
         "proximo-da-normalidade": {
            "en": "behavior close to normality",
            "es": "cerca de la normalidad"
        },
         "seco-ou-tendencia-a-seco": {
            "en": "dry behavior or a tending to be dry",
            "es": "seco o con tendencia a ser seco"
        }
    }
    return d[prognostico][lang]  

In [19]:
def get_analysis(doc, bulletin_dict):
    bacias = ['Bacia do Rio Branco', 'Bacia do Rio Negro','Bacia do Rio Marañon', 'Bacia do Rio Ucayali', 'Bacia do Rio Napo', 
'Curso principal do Rio Amazonas (Peru)','Bacia do Rio Javari','Bacia do Rio Içá (Putumayo)','Bacia do Rio Jutaí','Bacia do Rio Juruá',
'Bacia do Rio Japurá (Caquetá)','Bacia do Rio Tefé','Bacia do Rio Coari','Bacia do Rio Purus','Curso principal do Rio Solimões',
'Bacia dos rios Beni e Madre de Dios','Bacia do Rio Mamoré','Bacia do Rio Guaporé (Iténez)','Bacia do Rio Ji-Paraná','Bacia do Rio Aripuanã',
'Bacia do Rio Madeira','Bacias da margem esquerda do Rio Amazonas (Amazonas)','Bacia do Rio Abacaxis','Bacia do Rio Juruena','Bacia do Rio Teles Pires',
'Bacia do Rio Tapajós','Bacias da margem esquerda do Rio Amazonas (noroeste do Pará)','Bacia do Rio Curuá Una','Bacias da margem esquerda do Rio Amazonas (nordeste do PA)',
'Bacia do Rio Iriri','Bacia do Rio Xingu','Curso principal do Rio Amazonas (Brasil)']
    analysis = []
    texts = [doc.load_page(pg).get_text() for pg in range(4,15)]
    text = " ".join(texts)
    text = re.sub(r"\s*\n\s*", " ", text)
    text = re.sub(r"\s{2,}", " ", text).strip()
    for i, nome in enumerate(bacias):
        nome_regex = re.escape(nome)
        start_match = re.search(nome_regex, text)
        start = start_match.end()
        if i < len(bacias) - 1:
            next_nome = re.escape(bacias[i + 1])
            next_match = re.search(next_nome, text)
            end = next_match.start() if next_match else len(text)
        else:
            end = len(text)
        basin_text = text[start:end].strip()
        fields = extract_fields(basin_text)
        slug_name = slugify(nome)
        classification = fields["classification"]
        slug_classification = slugify(classification)
        # trend
        trend = fields["trend"]
        slug_trend = slugify(trend)
        # prognostico
        prognostico = fields["prognostico"]
        slug_prognostico = slugify(prognostico)

        
        bacia = {
                "id": slug_name,
                "name": nome,
                "min": fields["min"],
                "max": fields["max"],
                "observados": fields["observados"],
                "anomalia": fields["anomalia"],
                "i18n": {
                    "pt": {
                        "classification": classification,
                        "trend": trend,
                        "prognostico": prognostico
                    },
                    "en": {
                        "classification": dict_classes(slug_classification, "en"),
                        "trend": dict_trend(slug_trend, "en"),
                        "prognostico": dict_prognostico(slug_prognostico, "en")
                    },
                    "es": {
                        "classification": dict_classes(slug_classification, "es"),
                        "trend": dict_trend(slug_trend, "es"),
                        "prognostico": dict_prognostico(slug_prognostico, "es")
                    }
                }
            }
        # print(bacia)
        analysis.append(bacia)
    bulletin_dict['analysis'] = analysis
    
    return bulletin_dict

In [24]:
def get_multimodel(pathPT, pathEN, pathES, bulletin_dict):
    multimodel = {
        "pt": {},
        "en": {},
        "es": {}
    }
    # pt
    doc = pymupdf.open(pathPT)
    page = doc.load_page(15)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        if "A Figura acima," in text:
            seven_days = text.replace("\n", "")
            multimodel['pt']['seven_days'] = seven_days
            break
    page = doc.load_page(16)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        if "A Figura acima," in text:
            fourteen_days = text.replace("\n", "")
            multimodel['pt']['fourteen_days'] = fourteen_days
            break
    # en
    doc = pymupdf.open(pathEN)
    page = doc.load_page(15)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        text = re.sub(r'\s+', ' ', text).strip()
        if "The figure above"in text:
            seven_days = text.replace("\n", "")
            multimodel['en']['seven_days'] = seven_days
            break
    page = doc.load_page(16)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        text = re.sub(r'\s+', ' ', text).strip()
        if "The figure above" in text:
            fourteen_days = text.replace("\n", "")
            multimodel['en']['fourteen_days'] = fourteen_days
            break
    # es
    doc = pymupdf.open(pathES)
    page = doc.load_page(15)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        text = re.sub(r'\s+', ' ', text).strip()
        if "La figura de arriba"in text:
            seven_days = text.replace("\n", "")
            multimodel['es']['seven_days'] = seven_days
            break
    page = doc.load_page(16)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        text = re.sub(r'\s+', ' ', text).strip()
        if "La figura de arriba" in text:
            fourteen_days = text.replace("\n", "")
            multimodel['es']['fourteen_days'] = fourteen_days
            break

    bulletin_dict['multimodel'] = multimodel


    return bulletin_dict

In [20]:
def get_text(pathPT, mmdd):
    bulletin_dict = {}
    doc = pymupdf.open(pathPT)
    bulletin_dict = get_meta(doc, bulletin_dict)
    bulletin_dict = get_current_conditions(pathPT, pathEN, pathES, bulletin_dict)
    bulletin_dict = get_analysis(doc, bulletin_dict)
    bulletin_dict = get_multimodel(pathPT, pathEN, pathES, bulletin_dict)
    with open(f'/home/inacio/script-clima-amazonia/data/2026/{mmdd}.json', 'w') as f:
        json.dump(bulletin_dict, f, ensure_ascii=False, indent=3)
    return bulletin_dict
    



In [25]:
bulletin_dict = get_text(pathPT, "0304")

# Images

In [ ]:
def get_images(pathPT, output_path, edition):
    doc = pymupdf.open(pathPT)
    img_current_conditions(doc, output_path, edition)
    img_analysis(doc, output_path, edition)
    img_multimodel(doc, output_path, edition)
    img_reference(doc, output_path, edition)
    img_categorization(doc, output_path, edition)
    img_anomaly(doc, output_path, edition)

In [26]:
bulletin_dict

{'meta': {'pt': {'doi': '10.61818/02910609',
   'issn': '2965-0291',
   'volume': '06',
   'date': '4 de março de 2026'},
  'en': {'doi': '10.61818/78570409',
   'issn': '2965-7857',
   'volume': '04',
   'date': 'March 4, 2026'},
  'es': {'doi': '10.61818/77090409',
   'issn': '2965-7709',
   'volume': '04',
   'date': '4 de marzo de 2026'},
  'number': '09'},
 'current_conditions': {'pt': 'Mapas das condições observadas de precipitação e gráficos individuais por bacias são produzidos a partir dos dados MERGE/GPM gerados pelo INPE/CPTEC, considerando como climatologia para período de 2000 a 2025. Entre os dias 3 de fevereiro e 4 de março de 2026, chuvas abaixo da climatologia caracterizaram com déficit de precipitação o curso principal do Rio Amazonas em territórios brasileiro e peruno, bacias hidrográficas dos rios Abacaxis, Beni, Branco, Coari, Guaporé, Içá, Japurá, Javari, Ji-Paraná, Juruá, Juruena, Jutaí, Madeira, Mamoré, bacias da margem esquerda do Rio Amazonas no nordeste do Es

In [14]:
edition = '0304' 
output_path = "/home/inacio/script-clima-amazonia/data/images/"
# get_images(pathPT, output_path, edition)

In [4]:
config(
  cloud_name="dveg6vhbi",
  api_key="954532419258163",
  api_secret="MiQDehQG2afKCxVi-QC8qMFE0Ag"
)

In [6]:
result = uploader.upload(
    "/home/inacio/script-clima-amazonia/data/images/0304/current_conditions/map_current_conditions.png",
    public_id="map_current_conditions",
    folder="clima-amazonia/2026/fev/0225/current-conditions",
    use_filename=True,
)

In [13]:
result['secure_url']

'https://res.cloudinary.com/dveg6vhbi/image/upload/v1772820680/clima-amazonia/2026/fev/0225/current-conditions/map_current_conditions.jpg'

In [40]:
edition

'0304'

In [41]:
path = f'{output_path}{edition}'
d_imgs = {}
for section in os.listdir(path):
    full_path = f'{path}/{section}'
    d_imgs[section] = {}
    imgs = os.listdir(full_path)
    for img in imgs:
        full_img_path = f'{full_path}/{img}'
        img_name = img.split(".")[0]
        result = uploader.upload(
            full_img_path,
            public_id=img_name,
            folder=f"clima-amazonia/2026/fev/{edition}/{section}",
            use_filename=True,
        )
        url = result['secure_url']
        d_imgs[section][img_name] = url
        print(result)

{'asset_id': 'c27918d3ab5ff1535d39153cd74bb933', 'public_id': 'clima-amazonia/2026/fev/0304/current_conditions/table_current_conditions', 'version': 1772825245, 'version_id': '89b2f70ebb2e4f28e233e11f8c7b1fa2', 'signature': '6922c3503643b0dace0aaa5385f35c6099255561', 'width': 1349, 'height': 270, 'format': 'png', 'resource_type': 'image', 'created_at': '2026-03-06T19:27:25Z', 'tags': [], 'bytes': 60516, 'type': 'upload', 'etag': '782cf23e4ed78d1554aa027eff4ec431', 'placeholder': False, 'url': 'http://res.cloudinary.com/dveg6vhbi/image/upload/v1772825245/clima-amazonia/2026/fev/0304/current_conditions/table_current_conditions.png', 'secure_url': 'https://res.cloudinary.com/dveg6vhbi/image/upload/v1772825245/clima-amazonia/2026/fev/0304/current_conditions/table_current_conditions.png', 'asset_folder': 'clima-amazonia/2026/fev/0304/current_conditions', 'display_name': 'table_current_conditions', 'original_filename': 'table_current_conditions', 'api_key': '954532419258163'}
{'asset_id': '9

In [43]:
bulletin_dict['images'] = d_imgs

In [45]:
mmdd = '0304'
with open(f'/home/inacio/script-clima-amazonia/data/2026/{mmdd}.json', 'w') as f:
        json.dump(bulletin_dict, f, ensure_ascii=False, indent=3)